# Band edge: Figures 9, 10 and 11

The field response near the absorption edge, on the valley disks (all three field terms, $\nu = 5$ meV). Fig. 9 is $\eta(B)$ just above the gap, Fig. 10 the field coefficient over the first 0.45 eV, and Fig. 11 compares the threshold coefficient across models.

## Load engine

In [ ]:
%run engine.ipynb

## Settings

In [ ]:
R_disk = 0.9       # disk radius around each valley (1/a)
h_edge = 0.0007    # disk spacing in Fig. 9 (0.005 for a quick look)
dB_edge = 2        # field step in Fig. 9, in T; the LDA runs always use 10 T (10 for a quick look)
h_chi = 0.001      # disk spacing in Fig. 10 (0.002 for a quick look; coarser grids give false sign changes)
d_off = 0.001      # photon-energy step in Fig. 10, in eV (0.005 for a quick look)
N_kp = 401         # k.p grid across the disk for the squares of Fig. 11 (301 for a quick look)

valleys = [K_len, -K_len]

## Figure 9: $\eta^{x;xx}(B)$ at the band edge

At $E_g + 0.01$ eV, averaged over five photon energies from $E_g + 0.0075$ to $E_g + 0.0125$ eV. The threshold coefficient $\chi$ is the slope $s$ of $\eta = sB + cB^3$ fitted to the data (with $\eta(-B) = -\eta(B)$). Fig. 11 also needs the NN model and the LDA parameters, so they are run here too. This is the slowest part: about 13 min per field at $h = 0.0007/a$ (about $10^7$ points), 55 h for the eight GGA curves and 10 h for the eight LDA ones on one core.

In [ ]:
edge_disk = disk(R_disk, h_edge)
edge = {}
for func, dB in [("GGA", dB_edge), ("LDA", 10)]:
    B_edge = np.arange(0, 60 + 1e-9, dB)
    for model in ["TNN", "NN"]:
        for m in materials:
            file = f"data/edge_{m}_{model}_{func}_R{R_disk}_h{h_edge}_dB{dB}.npz"
            if not os.path.exists(file):
                omega = gap(m, model, func) + 0.01 + np.linspace(-0.0025, 0.0025, 5)
                eta = []
                for B in B_edge:
                    eta.append(eta_mag(m, model, B, omega, 0.005, edge_disk, centers=valleys, func=func)[:, 0, 0, 0].mean())
                np.savez(file, B=B_edge, eta=eta_SI(params(m, func)["a"]) * np.array(eta))
            d = np.load(file)
            B_all, eta_all = mirror(d["B"], d["eta"])
            s, c = np.linalg.lstsq(np.stack([B_all, B_all**3], axis=1), eta_all, rcond=None)[0]
            edge[m, model, func] = dict(B=B_all, eta=eta_all, chi=s, cubic=100 * c * 60**2 / s)

for func in ["GGA", "LDA"]:
    for model in ["TNN", "NN"]:
        print(f"{func} {model:3s} chi (1e-4 A m V^-2 s^-1 T^-1):", np.round([1e4 * edge[m, model, func]["chi"] for m in materials], 3),
              " cubic part at 60 T (%):", np.round([edge[m, model, func]["cubic"] for m in materials], 2))

In [ ]:
scale = int(np.floor(np.log10(max(np.abs(edge[m, "TNN", "GGA"]["eta"]).max() for m in materials))))
fig, axes = plt.subplots(1, 4, figsize=(TW, 1.85), gridspec_kw=dict(wspace=0.42))
for j, m in enumerate(materials):
    d = edge[m, "TNN", "GGA"]
    ax = axes[j]
    ax.plot(d["B"], d["eta"] / 10.0**scale, color=RED if d["chi"] > 0 else BLUE)
    zero(ax, v=True)
    ax.set_xlabel(r"$B$ (T)")
    ax.set_xlim(-62, 62)
    ax.xaxis.set_major_locator(MaxNLocator(5, steps=[1, 2, 3, 5, 6, 10]))
    ax.set_title(mat_label[m], pad=3, loc="right")
    label(ax, f"({'abcd'[j]})")
    ax.yaxis.set_major_locator(MaxNLocator(5))
axes[0].set_ylabel(r"$\eta^{x;xx}$ ($10^{%d}\,%s$)" % (scale, UNIT))
plt.savefig("figures/lpge_edge_sign_grid.pdf")
plt.show()

## Figure 10: field coefficient against photon energy

$\chi(\omega) = \eta^{x;xx}(30\,{\rm T})/30$ T for $\hbar\omega - E_g$ from $-0.03$ to $0.45$ eV, TNN and NN, smoothed by a 5 meV running mean. Circles mark sign changes, and the dotted line is the onset $E_g + 2\lambda$ of the transition from the lower spin-split valence band (B). About 10 min per curve, 80 min in all.

In [ ]:
chi_disk = disk(R_disk, h_chi)
offset = np.arange(-0.03, 0.45 + 1e-9, d_off)
chi_w = {}
for model in ["TNN", "NN"]:
    for m in materials:
        file = f"data/chi_omega_{m}_{model}_R{R_disk}_h{h_chi}_d{d_off}.npz"
        if not os.path.exists(file):
            eta = eta_mag(m, model, 30.0, gap(m, model) + offset, 0.005, chi_disk, centers=valleys)[:, 0, 0, 0]
            np.savez(file, off=offset, chi=eta_SI(params(m)["a"]) * eta / 30)
        chi_w[m, model] = np.load(file)["chi"]

In [ ]:
n_avg = max(1, int(round(0.005 / d_off)))         # points in the 5 meV running mean
shown = (offset >= -0.02) & (offset <= 0.447)
fig, axes = plt.subplots(1, 4, figsize=(TW, 2.25), gridspec_kw=dict(wspace=0.36))
for j, m in enumerate(materials):
    ax = axes[j]
    curves = []
    for model, color in [("TNN", RED), ("NN", ORANGE)]:
        y = np.convolve(1e4 * chi_w[m, model], np.ones(n_avg) / n_avg, "same")[shown]
        x = offset[shown]
        ax.plot(x, y, color=color, lw=1.0, label=model)
        curves.append(y)

        # sign changes, by linear interpolation between neighbouring points
        k = np.where(np.sign(y[1:]) * np.sign(y[:-1]) < 0)[0]
        x0 = x[k] - y[k] * (x[k + 1] - x[k]) / (y[k + 1] - y[k])
        x0 = x0[x0 > 0.004]
        ax.plot(x0, 0 * x0, "o", ms=4, mfc="white", mec=color, mew=1.0, zorder=5, ls="none")
        print(f"{m} {model}: chi(Eg + 0.01) = {np.interp(0.01, x, y):+.3f} x 1e-4, sign changes at Eg +", np.round(x0, 3), "eV")
    all_y = np.concatenate(curves)
    lo, hi = min(all_y.min(), 0), max(all_y.max(), 0)
    ax.set_ylim(lo - 0.08 * (hi - lo), hi + 0.08 * (hi - lo))
    ax.axvline(0, color="0.6", lw=0.6)
    if 2 * lambda_SOC[m] < 0.45:
        ax.axvline(2 * lambda_SOC[m], color="0.5", lw=0.6, ls=":")
        ax.text(2 * lambda_SOC[m] - 0.008, 0.93, "B", transform=ax.get_xaxis_transform(), fontsize=7, color="0.35", ha="right")
    zero(ax)
    ax.set_xlim(-0.02, 0.45)
    ax.set_xlabel(r"$\hbar\omega-E_g$ (eV)")
    ax.yaxis.set_major_locator(MaxNLocator(6))
    ax.set_title(mat_label[m], pad=3, loc="right")
    label(ax, f"({'abcd'[j]})")
axes[0].set_ylabel(r"$\chi$ ($10^{-4}\,%s\,\mathrm{T^{-1}}$)" % UNIT)
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.07), frameon=False)
plt.savefig("figures/chi_vs_offset.pdf")
plt.show()

## Figure 11: threshold field coefficient across models

Tight-binding $\chi$ from the fits of Fig. 9 (GGA filled, LDA open), against the material in (a) and against $\kappa$ from the Löwdin reduction in (b), with a straight-line fit to the eight GGA points.

The squares are the k.p model with the first-principles parameters of Kormányos *et al.*, 2D Mater. **2**, 022001 (2015), for the band-edge spin pair, with $k$ in 1/Å: $m_D = (E_A + \lambda)/2$ and $w = (\hbar^2/4m_e)(1/m_c + 1/|m_v|) - v^2/E_A$. Their $\chi = \eta^{x;xx}(30\,{\rm T})/30$ T at $E_A + 0.01$ eV, on a disk of radius $0.2$/Å with $\nu = 5$ meV. A few seconds.

In [ ]:
#          E_A    v      kappa  m_c    m_v     (eV, eV A, eV A^2, m_e)
korm = {"MoS2":  (1.670, 2.76, -1.45, 0.43, -0.54),
        "MoSe2": (1.400, 2.53, -1.23, 0.49, -0.59),
        "WS2":   (1.632, 3.34, -1.09, 0.26, -0.35),
        "WSe2":  (1.337, 3.17, -0.93, 0.28, -0.36)}
hbar2_2m = hbar**2 / (2 * m_e) / e * 1e20      # hbar^2 / 2 m_e in eV A^2 (3.80998)

chi_kp = []
for m in materials:
    E_A, v, kap, m_c, m_v = korm[m]
    lam = lambda_SOC[m]
    kp_fp = dict(v=v, kappa=kap, w=0.5 * hbar2_2m * (1 / m_c + 1 / abs(m_v)) - v**2 / E_A, mD=(E_A + lam) / 2, lam=lam)
    eta = kp_currents(kp_fp, mu_B * 30, np.array([E_A + 0.01]), 0.005, 0.2, N_kp)
    chi_kp.append(1e4 * eta_SI(1.0) * eta[0, 0, 0, 0] / 30)        # k in 1/A, so a = 1 A
print("k.p chi (1e-4 A m V^-2 s^-1 T^-1):", np.round(chi_kp, 3))
kappa_fp = [korm[m][2] for m in materials]

kappa = {}
for func in ["GGA", "LDA"]:
    for model in ["TNN", "NN"]:
        for m in materials:
            r = lowdin(m, func, model)
            kappa[m, model, func] = r["kappa"] * r["a"]**2

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(TW, 2.35), gridspec_kw=dict(wspace=0.28))
marks = lambda color, filled, name: dict(color=color, mfc=color if filled else "none", mew=0.9, ms=4.5, ls="none", label=name)
x_GGA, y_GGA = [], []
styles = [("TNN", "o", RED), ("NN", "^", ORANGE)]
for dx, (model, marker, color) in zip([-0.16, 0], styles):
    chi = [1e4 * edge[m, model, "GGA"]["chi"] for m in materials]
    kap = [kappa[m, model, "GGA"] for m in materials]
    axes[0].plot(np.arange(4) + dx, chi, marker, **marks(color, True, "TB, " + model))
    axes[1].plot(kap, chi, marker, **marks(color, True, "TB, " + model))
    x_GGA += kap
    y_GGA += chi
kp_name = r"$\mathbf{k}\cdot\mathbf{p}$, first-principles"
axes[0].plot(np.arange(4) + 0.16, chi_kp, "s", **marks(BLUE, False, kp_name))
axes[1].plot(kappa_fp, chi_kp, "s", **marks(BLUE, False, kp_name))
kappa_LDA = {model: [kappa[m, model, "LDA"] for m in materials] for model in ["TNN", "NN"]}
for model, marker, color in styles:
    axes[1].plot(kappa_LDA[model], [1e4 * edge[m, model, "LDA"]["chi"] for m in materials], marker, **marks(color, False, f"TB, {model} (LDA)"))

slope, intercept = np.polyfit(x_GGA, y_GGA, 1)
r = np.corrcoef(x_GGA, y_GGA)[0, 1]
all_kappa = x_GGA + kappa_fp + kappa_LDA["TNN"] + kappa_LDA["NN"]
x_line = np.linspace(min(all_kappa) - 0.12, max(all_kappa) + 0.08, 50)
axes[1].plot(x_line, slope * x_line + intercept, ":", color="0.4", lw=0.9, label="fit to TB (GGA)")
print(f"Straight-line fit to the 8 GGA TB points: chi = ({slope:.3f} kappa {intercept:+.3f}) x 1e-4, r = {r:.3f}")

axes[0].set_xticks(range(4), [mat_label[m] for m in materials])
axes[0].tick_params(axis="x", which="minor", bottom=False, top=False)
axes[0].set_xlim(-0.5, 3.5)
all_chi = y_GGA + list(chi_kp)
lo, hi = min(min(all_chi), 0), max(max(all_chi), 0)
axes[0].set_ylim(lo - 0.06 * (hi - lo), hi + 0.45 * (hi - lo))
axes[0].set_ylabel(r"$\chi(E_g+0.01\,\mathrm{eV})$ ($10^{-4}\,%s\,\mathrm{T^{-1}}$)" % UNIT)
axes[1].set_xlabel(r"$\kappa$ (eV$\,\mathrm{\AA}^2$)")
for i, ax in enumerate(axes):
    zero(ax, v=(i == 1))
    label(ax, f"({'ab'[i]})")
axes[0].legend(loc="upper left")
axes[1].legend(loc="upper right")
plt.savefig("figures/threshold_models.pdf")
plt.show()

## Convergence in $h$

$\eta^{x;xx}(30\,{\rm T})$ at threshold with the disk spacing $h = 0.001/a$, against the $h = 0.0007/a$ of Fig. 9. About 1 h.

In [ ]:
file = "data/convergence_h.npz"
if not os.path.exists(file):
    coarse = disk(R_disk, 0.001)
    out = {}
    for model in ["TNN", "NN"]:
        for m in materials:
            omega = gap(m, model) + 0.01 + np.linspace(-0.0025, 0.0025, 5)
            out[f"{m}_{model}"] = eta_SI(params(m)["a"]) * eta_mag(m, model, 30.0, omega, 0.005, coarse, centers=valleys)[:, 0, 0, 0].mean()
    np.savez(file, **out)
d = np.load(file)
for model in ["TNN", "NN"]:
    for m in materials:
        fine = edge[m, model, "GGA"]["eta"][edge[m, model, "GGA"]["B"] == 30][0]
        print(f"{m} {model}: eta(30 T) changes by {100 * (d[f'{m}_{model}'] / fine - 1):+.3f}% from h = 0.0007 to h = 0.001")

## Size of the orbital terms

Three numbers the text quotes. The itinerant moment $|M_n|$ of the valence and lowest conduction bands at $K$ and $K'$. The largest change of the phase-space weight, $|D_n - 1| = (ea^2/\hbar)\,|B\,\Omega_n|$, for those bands at 60 T on disks of radius $0.3/a$. And how much the phase-space factor changes the threshold coefficient: $|\chi_{ED}/\chi_E - 1|$, where E has the energy shift alone and ED adds $D_v$, at $B = 10$ T on the grid of Fig. 9 with $h = 0.002/a$. About 5 min.

In [ ]:
moments, D_shift = [], []
near_K_x, near_K_y, _ = disk(0.3, 0.01)
for m in materials:
    p = params(m)
    for k_x0, s in [(K_len, 0), (-K_len, 1)]:
        H = lambda x, y: zeeman(hamiltonian("TNN", p, x, y), lambda_SOC[m], 0.0)[s]
        E, V, v_x, v_y = bands(H, np.array([k_x0]), np.array([0.0]))
        moments += list(np.abs(berry_and_moment(E, v_x, v_y, p["a"])[1][0, :2]))
    for k_x0 in valleys:
        for s in (0, 1):
            H = lambda x, y: zeeman(hamiltonian("TNN", p, x, y), lambda_SOC[m], 60.0)[s]
            E, V, v_x, v_y = bands(H, k_x0 + near_K_x, near_K_y)
            Omega = berry_and_moment(E, v_x, v_y, p["a"])[0]
            D_shift.append(np.abs(e * (p["a"] * 1e-10)**2 / hbar * 60.0 * Omega[:, :2]).max())
print(f"|M_n| at K: {min(moments):.2f} to {max(moments):.2f} mu_B")
print(f"largest |D_n - 1| at 60 T: {100 * max(D_shift):.1f}%")

file = f"data/phase_space_R{R_disk}_h0.002.npz"
if not os.path.exists(file):
    threshold_disk = disk(R_disk, 0.002)
    out = {}
    for m in materials:
        omega = gap(m) + 0.01 + np.linspace(-0.0025, 0.0025, 5)
        for name, D_on in [("E", False), ("ED", True)]:
            out[f"{m}_{name}"] = eta_mag(m, "TNN", 10.0, omega, 0.005, threshold_disk, centers=valleys,
                                         velocity_shift=False, phase_space=D_on)[:, 0, 0, 0].mean()
    np.savez(file, **out)
d = np.load(file)
change = {m: 100 * abs(d[m + "_ED"] / d[m + "_E"] - 1) for m in materials}
for m in materials:
    print(f"{m}: the phase-space factor changes the threshold chi by {change[m]:.1f}%")
print(f"range: {min(change.values()):.1f}% to {max(change.values()):.1f}%")